# Entrenamiento de YOLO11 en VisDrone
## Paso previo OBLIGATORIO para el benchmark de mAP — FLISOL 2026

### ¿Por qué entrenar? (el bug del mAP ~0)

Si evalúas con `yolo11s.pt` (pre-entrenado en **COCO, 80 clases**) contra el ground truth
de **VisDrone (10 clases)**, los `category_id` **no coinciden**. COCOeval compara clases
distintas → el mAP sale casi **0** (≈0.002). Eso es *zero-shot*, no un mal desempeño de SAHI.

| | Modelo COCO (zero-shot) | Modelo entrenado en VisDrone |
|---|---|---|
| Clases | 80 (person, car, ...) | 10 (pedestrian, people, ..., motor) |
| Alineación con GT VisDrone | ❌ no | ✅ sí |
| mAP esperado | ~0.00 | ~0.20–0.35 |

Este notebook entrena el modelo y guarda `best.pt`. Ejecútalo **antes de la charla**
(en T4, ~2–4 h según épocas). Luego usa `02_resultados_figuras.ipynb` para el benchmark.

> ⚙️ Activa la GPU (T4) antes de empezar.

---
## 1. Setup

In [ ]:
!pip install -q ultralytics sahi
!git clone https://github.com/jossuema/yolo-sahi-flisol2026.git || echo 'ya clonado'
%cd yolo-sahi-flisol2026
import torch; print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU')

---
## 2. (Recomendado) Montar Google Drive para PERSISTIR los pesos

Colab borra el disco al desconectarse. Guardar `best.pt` en Drive evita reentrenar.
Si prefieres no usar Drive, salta esta celda y descarga `best.pt` manualmente al final.

In [ ]:
USE_DRIVE = True
DRIVE_DIR = '/content/drive/MyDrive/flisol2026'
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    import os; os.makedirs(DRIVE_DIR, exist_ok=True)
    print('Pesos se guardarán en:', DRIVE_DIR)

---
## 3. Entrenar

Ultralytics **descarga y convierte VisDrone automáticamente** (train ~1.4 GB).

Parámetros:
- `--epochs`: 50 da buenos resultados; usa `3` para una prueba rápida del pipeline.
- `--imgsz`: 640 estándar. **960/1024 mejora objetos pequeños** (más lento, más VRAM).
- `--batch`: 16, o `-1` para autobatch.

Esto tarda. Déjalo corriendo.

In [ ]:
# Prueba rápida (valida que todo corre). Cambia a 50 épocas para el resultado real.
!python src/train_visdrone.py --model yolo11s.pt --epochs 3 --imgsz 640 --batch 16

In [ ]:
# Entrenamiento REAL (descomenta y ejecuta cuando vayas a entrenar de verdad)
# !python src/train_visdrone.py --model yolo11s.pt --epochs 50 --imgsz 640 --batch 16

---
## 4. Validación rápida del modelo

Confirmamos que el modelo tiene **10 clases VisDrone** (no 80 COCO) y vemos su mAP de validación nativo.

In [ ]:
from ultralytics import YOLO
m = YOLO('weights/visdrone_yolo11s.pt')
print('Clases del modelo:', m.names)
assert len(m.names) == 10, 'El modelo NO tiene 10 clases VisDrone; algo falló.'
metrics = m.val(data='VisDrone.yaml', imgsz=640)
print('mAP50-95:', round(metrics.box.map, 4), '| mAP50:', round(metrics.box.map50, 4))

---
## 5. Guardar `best.pt` (Drive y/o descarga)

In [ ]:
import shutil, os
SRC = 'weights/visdrone_yolo11s.pt'
if USE_DRIVE:
    dst = os.path.join(DRIVE_DIR, 'visdrone_yolo11s.pt')
    shutil.copy(SRC, dst); print('Guardado en Drive:', dst)
try:
    from google.colab import files; files.download(SRC)
except Exception as e:
    print('Descarga manual:', SRC, e)

---
## 6. Siguiente paso

Con `weights/visdrone_yolo11s.pt` listo:

- **`02_resultados_figuras.ipynb`** → benchmark mAP real (YOLO vs YOLO+SAHI) + todas las figuras.
- **`01_demo_colab.ipynb`** → demo en vivo (puedes usar este modelo o el COCO para lo visual).

En esos notebooks, monta Drive y copia el peso a `weights/` antes de evaluar:
```python
from google.colab import drive; drive.mount('/content/drive')
!mkdir -p weights && cp /content/drive/MyDrive/flisol2026/visdrone_yolo11s.pt weights/
```

---
### Avanzado (opcional): *sliced fine-tuning*
SAHI permite **cortar el dataset en slices** y entrenar sobre ellos (`sahi coco slice`).
Entrenar en slices + inferir con SAHI suele exprimir aún más el AP de objetos pequeños.
Para la charla, el fine-tuning estándar de arriba ya es suficiente.